# OP-FL v2 — 10. Validation interne d'OP-Cal v2 (spécification 0.3 gelée)

**Périmètre.**
* Tâche : mortalité, modèle `x_only`, sur les **60 hôpitaux de développement uniquement**.
* Aucun label, aucune prédiction et aucun descripteur des 96 hôpitaux verrouillés n'est lu. Du manifest, on ne lit que les effectifs `n_train`, `n_val` et `n_test` des hôpitaux verrouillés, pour fixer la taille des sous-échantillons, comme au 08.
* Le bras mécanistique (lactate, taux de dosage ajusté) fera l'objet du notebook 11.

**Ce que le notebook applique** (spécification, sections 5 à 7) :
* les **méthodes** : E0, E1, v1, v1 + rétrécissement fixe, v1 + bornage, v1 + débruitage, GP sans débruitage et **v2** ;
* les **schémas** :
  * S1 : plis par hôpital (protocole + 10 partitions) ;
  * S2 : retrait d'un groupe de case-mix (k = 5, 10 graines) ;
  * S3 : S2 avec sous-échantillonnage à la taille des hôpitaux verrouillés (20 tirages) ;
* une **observation par hôpital** : moyenne sur les répétitions où il est tenu à l'écart ;
* le **bootstrap du maximum** : 2000 tirages d'hôpitaux, maximum des comparateurs recalculé à chaque tirage.

**Règle de décision** (fixée par la spécification). La v2 est retenue si les trois conditions suivantes sont réunies :
* S3 : la borne basse de l'IC de D = Δℓ(v2) − max(E1, v1, rétrécissement fixe, bornage) est > 0 ;
* S2 : D > 0 ;
* S1 : la borne basse de l'IC de Δℓ(v2) − Δℓ(v1) est > −0,0005.

Si la v2 est retenue, la validation sur les 49 petits hôpitaux suivra, avec un plan commité avant.

**Durée** : environ 5 à 15 minutes (105 plis, chacun avec deux GP et huit méta-modèles ; aucun entraînement de réseau).

In [1]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION, INTÉGRITÉ, PLAN
# ============================================================
import sys, json, time, hashlib, importlib, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_DIR = Path("..")
DATA_DIR, SRC_DIR = PROJECT_DIR / "data_v1", PROJECT_DIR / "src"
RES02, RES04, RES06B, RES06C = (PROJECT_DIR / "results" / d for d in ("02_baselines", "04_observation_label", "06b_faisabilite_opcal", "06c_robustesse_opcal"))
RES_DIR = PROJECT_DIR / "results" / "10_validation_v2"
RES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR.resolve()))
sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

protocol = json.loads((DATA_DIR / "splits" / "protocol.json").read_text(encoding="utf-8"))
assert protocol["hash"] == "d92099ea7c11d517"
FOLDS = sorted(int(k) for k in protocol["folds"])
DEV = sorted(int(h) for k in protocol["folds"] for h in protocol["folds"][k])
LOCKED = set(int(h) for h in protocol["locked_test_hospitals"])
FROZEN = json.loads((RES06C / "specification_gelee.json").read_text(encoding="utf-8"))["meta_modele"]["descripteurs"]
GP_SHA = sha256(SRC_DIR / "opcal_gp.py")

SEED, N_PART, K_CLUST, N_SEEDS, N_DRAW, N_BOOT = 2026, 10, 5, 10, 20, 2000
PLAN = {"document": "Plan du notebook 10 (validation interne OP-Cal v2, spécification 0.3)",
        "tache": "mortalité, x_only, 60 hôpitaux de développement", "opcal_gp_sha256": GP_SHA,
        "schemas": {"S1": f"plis du protocole + {N_PART} partitions (06d)", "S2": f"k-moyennes k={K_CLUST} sur le case-mix, {N_SEEDS} graines",
                    "S3": f"S2 + sous-échantillonnage à la taille des verrouillés, {N_DRAW} tirages"},
        "comparateurs_du_maximum": ["E1", "v1", "v1_retreci", "v1_borne"], "bootstrap": N_BOOT, "graine": SEED,
        "decision": {"S3": "borne basse IC de D > 0", "S2": "D > 0", "S1": "borne basse IC de Δℓ(v2) − Δℓ(v1) > −0,0005"}}
PLAN_PATH = RES_DIR / "plan_10.json"
if PLAN_PATH.exists():
    old = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
    assert {k: v for k, v in old.items() if k != "date"} == PLAN, "plan_10.json existe avec un autre contenu (module ou plan modifié)."
    PLAN = old
else:
    PLAN["date"] = datetime.now().isoformat(timespec="seconds")
    PLAN_PATH.write_text(json.dumps(PLAN, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"opcal_gp.py {GP_SHA[:16]}… ; plan {PLAN_PATH.name} du {PLAN['date']}")

opcal_gp.py 052802442a27c1f7… ; plan plan_10.json du 2026-10-31T21:24:54


## Étape 1 — Module `src/opcal_v2eval.py`

In [2]:
%%writefile ../src/opcal_v2eval.py
"""
opcal_v2eval — validation interne d'OP-Cal v2 (notebook 10), hôpitaux de développement uniquement.

Généré par notebooks/10_validation_v2.ipynb (étape 1) : modifier le notebook, pas ce fichier.
Spécification : OP-Cal_v2_specification.md, révision 0.3 (sections 5 à 7).

- fit_methods : les 8 méthodes de la section 6, apprises sur les sites d'entraînement d'un pli ; renvoie un
  prédicteur pour des sites tenus à l'écart (descripteurs ẑ et variances d'échantillonnage C, sans label).
- run_scheme : plis × (taille complète ou sous-échantillonnage) ; une observation par hôpital (moyenne sur les
  répétitions où il est tenu à l'écart).
- decide : bootstrap des hôpitaux avec maximum des comparateurs recalculé à chaque tirage.
"""
import numpy as np
import pandas as pd

import opcal_feas as oc
import opcal_amend as am
import opcal_size as osz
from opcal_gp import SiteGP

METHODS = ["E0", "E1", "v1", "v1_retreci", "v1_borne", "v1_debruite", "GP_sans_debruitage", "v2"]
COMPARATEURS = ["E1", "v1", "v1_retreci", "v1_borne"]          # maximum de la règle de décision (section 7)
KAPPA = np.round(np.linspace(0, 1, 11), 2)                     # grille du rétrécissement fixe
CASE_PREFIX = ("age_mean", "female_share", "src_", "dx_")


def site_gain(lp, y, a):
    return am.loglik(lp + a, y) - am.loglik(lp, y)


def kmeans(X, k, seed, n_init=10, iters=200):
    """k-moyennes (initialisation k-means++), meilleure inertie sur n_init départs ; sans label."""
    rng = np.random.default_rng(seed)
    best, best_in = None, np.inf
    for _ in range(n_init):
        cen = [X[rng.integers(len(X))]]
        for _ in range(1, k):
            d2 = np.min(((X[:, None] - np.array(cen)[None]) ** 2).sum(2), 1)
            cen.append(X[rng.choice(len(X), p=d2 / d2.sum())])
        cen = np.array(cen)
        for _ in range(iters):
            lab = np.argmin(((X[:, None] - cen[None]) ** 2).sum(2), 1)
            new = np.array([X[lab == j].mean(0) if (lab == j).any() else cen[j] for j in range(k)])
            if np.allclose(new, cen):
                break
            cen = new
        inertia = ((X - cen[lab]) ** 2).sum()
        if inertia < best_in and len(np.unique(lab)) == k:
            best, best_in = lab, inertia
    return best


def _denoise(Ztr, Ctr, Z, C):
    """Débruitage diagonal (spécification 2.A) en unités d'origine : z̄ = m + λ (ẑ − m), λ = s²/(s² + C)."""
    with np.errstate(all="ignore"):
        fill = np.nanmean(Ztr, 0)
    fill = np.where(np.isfinite(fill), fill, 0.0)
    Zt = np.where(np.isfinite(Ztr), Ztr, fill)
    m, var = Zt.mean(0), Zt.var(0)
    s2 = np.maximum(var - np.nan_to_num(Ctr).mean(0), 0.05 * np.maximum(var, 1e-12))
    out = lambda Zx, Cx: m + s2 / (s2 + np.nan_to_num(Cx)) * (np.where(np.isfinite(Zx), Zx, fill) - m)
    return out(Ztr, Ctr), out(Z, C) if Z is not None else None, out


def fit_methods(Ztr, Ctr, c, v, ev_tr, inner_seed):
    """Apprend les méthodes sur les sites d'entraînement. ev_tr : liste (lp, y) par site d'entraînement (pour le
    réglage du rétrécissement fixe, validation croisée interne à 5 plis). Renvoie (predict, info)."""
    e1 = oc.SiteEB().fit(np.zeros((len(c), 0)), c, v)
    v1 = oc.SiteEB().fit(Ztr, c, v)
    # rétrécissement fixe : κ réglé par validation croisée interne sur le gain de log-vraisemblance
    rng = np.random.default_rng(inner_seed)
    inner = np.empty(len(c), int)
    inner[rng.permutation(len(c))] = np.arange(len(c)) % 5
    gains = np.zeros(len(KAPPA))
    for g in range(5):
        a_, b_ = inner != g, inner == g
        mi = oc.SiteEB().fit(Ztr[a_], c[a_], v[a_])
        ahat = mi.predict(Ztr[b_])[0]
        idx = np.flatnonzero(b_)
        for k_, kap in enumerate(KAPPA):
            gains[k_] += sum(site_gain(*ev_tr[i], mi.mu + kap * (ah - mi.mu)) for i, ah in zip(idx, ahat))
    kappa = float(KAPPA[int(np.argmax(gains))])
    lo, hi = np.nanmin(Ztr, 0), np.nanmax(Ztr, 0)
    Zd_tr, _, den = _denoise(Ztr, Ctr, None, None)
    v1d = oc.SiteEB().fit(Zd_tr, c, v)
    gp0 = SiteGP(denoise=False).fit(Ztr, Ctr, c, v)
    gp = SiteGP().fit(Ztr, Ctr, c, v)
    info = {"kappa": kappa, "gp_converge": gp.converged, "gp_borne": gp.at_bound, "gp0_converge": gp0.converged,
            "tau2": gp.tau2, "s2L": gp.s2L, "s2R": gp.s2R, "ellR": gp.ellR, "ellL": gp.ell_L}

    def predict(Z, C):
        a1 = v1.predict(Z)[0]
        out = {"E0": np.zeros(len(Z)), "E1": np.full(len(Z), e1.mu), "v1": a1,
               "v1_retreci": v1.mu + kappa * (a1 - v1.mu),
               "v1_borne": v1.predict(np.where(np.isfinite(Z), np.clip(Z, lo, hi), Z))[0],
               "v1_debruite": v1d.predict(den(Z, C))[0],
               "GP_sans_debruitage": gp0.predict(Z, C)[0] if gp0.converged else a1,      # repli fixé à l'avance
               "v2": gp.predict(Z, C)[0] if gp.converged else a1}
        return out
    return predict, info


def run_scheme(arr, cal, cols, fold_sets, pairs=None, modes=(0,), seed=2026, log=print):
    """fold_sets : liste de tableaux de plis alignés sur cal.index. modes : nombres de tirages ; 0 = taille complète,
    k > 0 = k sous-échantillonnages des sites tenus à l'écart à la taille des hôpitaux verrouillés (tirages du 08).
    Les méthodes sont apprises une fois par pli et servent à tous les modes.
    Renvoie ({mode: par_hopital}, infos) ; par_hopital : moyennes par hôpital de G, n et de l'erreur débiaisée."""
    hosp = cal.index.to_numpy()
    H, M = len(hosp), len(METHODS)
    site = [arr.site(h) for h in hosp]
    Zf = np.array([s_[0] for s_ in site]); Cf = np.array([np.diag(s_[1]) for s_ in site])
    c, v = cal["citl"].to_numpy(), cal["var_citl"].to_numpy()
    evf = [arr.eval(h) for h in hosp]
    acc = {md: {"G": np.zeros((H, M)), "N": np.zeros(H), "SQ": np.zeros((H, M)), "K": np.zeros(H)} for md in modes}
    infos = []
    for r, fold in enumerate(fold_sets):
        preds = {}
        for f in np.unique(fold):
            tr = np.flatnonzero(fold != f)
            p_, inf = fit_methods(Zf[tr], Cf[tr], c[tr], v[tr], [evf[i] for i in tr], seed + 1000 * r + int(f))
            preds[f] = p_
            infos.append({"repetition": r, "pli": int(f), **inf})
        for md in modes:
            a_ = acc[md]
            for d in ([None] if md == 0 else range(md)):
                if d is None:
                    Z, C, tg, ev = Zf, Cf, np.column_stack([c, v]), evf
                else:
                    rng = np.random.default_rng([seed, r, d])
                    Zs, Cs, tg, (lp, y, row), _ = osz._draw(arr, hosp, pairs, 1.0, osz.SCENARIOS["tout"], rng)
                    Z, C = Zs, np.array([np.diag(x) for x in Cs])
                    ev = [(lp[row == i], y[row == i]) for i in range(H)]
                for f, pr in preds.items():
                    te = np.flatnonzero(fold == f)
                    out = pr(Z[te], C[te])
                    for j, mth in enumerate(METHODS):
                        for i, a in zip(te, out[mth]):
                            a_["G"][i, j] += site_gain(*ev[i], a)
                            a_["SQ"][i, j] += (tg[i, 0] - a) ** 2 - tg[i, 1]
                    for i in te:
                        a_["N"][i] += len(ev[i][0]); a_["K"][i] += 1
        log(f"  répétition {r + 1}/{len(fold_sets)} terminée")
    ev_n = np.array([e[1].sum() for e in evf])
    per = {md: {"G": a_["G"] / a_["K"][:, None], "n": a_["N"] / a_["K"], "SQ": a_["SQ"] / a_["K"][:, None], "evenements": ev_n}
           for md, a_ in acc.items()}
    return per, pd.DataFrame(infos)


def scheme_table(per, min_events=10):
    """Δℓ (poids patients et poids égaux par hôpital) et erreur quadratique débiaisée, par méthode."""
    G, n = per["G"], per["n"]
    keep = per["evenements"] >= min_events
    return pd.DataFrame({"Δℓ (par patient)": G.sum(0) / n.sum(), "Δℓ (moyenne par hôpital)": (G / n[:, None]).mean(0),
                         "MSE débiaisé": per["SQ"][keep].mean(0)}, index=METHODS)


def boot_max(per, n_boot=2000, seed=2026):
    """D = Δℓ(v2) − max des comparateurs, maximum recalculé à chaque tirage ; et Δℓ(v2) − Δℓ(v1)."""
    G, n = per["G"], per["n"]
    j = {m: i for i, m in enumerate(METHODS)}
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(n), (n_boot, len(n)))
    dl = G[idx].sum(1) / n[idx].sum(1)[:, None]                                     # (B, M)
    Dmax = dl[:, j["v2"]] - dl[:, [j[m] for m in COMPARATEURS]].max(1)
    Dv1 = dl[:, j["v2"]] - dl[:, j["v1"]]
    full = G.sum(0) / n.sum()
    est_max = full[j["v2"]] - max(full[j[m]] for m in COMPARATEURS)
    return {"D_max": est_max, "D_max_bas": np.quantile(Dmax, 0.025), "D_max_haut": np.quantile(Dmax, 0.975),
            "meilleur_comparateur": max(COMPARATEURS, key=lambda m: full[j[m]]),
            "D_v1": full[j["v2"]] - full[j["v1"]], "D_v1_bas": np.quantile(Dv1, 0.025), "D_v1_haut": np.quantile(Dv1, 0.975)}

Writing ../src/opcal_v2eval.py


## Étape 2 — Données du développement et contrôle de reproduction (comme au 08)

In [3]:
# ============================================================
# ÉTAPE 2 — DONNÉES DU DÉVELOPPEMENT
# ============================================================
import opcal_feas as oc, opcal_size as osz, opcal_gp as ogp, opcal_v2eval as ov
for m_ in (oc, osz, ogp, ov):
    importlib.reload(m_)
st = pd.read_parquet(RES04 / "stay_labels.parquet").set_index("stay_id")
assert set(st["hospitalid"]) <= set(DEV)
coh = pd.read_parquet(DATA_DIR / "interim" / "cohort_final.parquet",
                      columns=["patientunitstayid", "hospitalid", "age_num", "gender", "apacheadmissiondx", "unittype", "unitadmitsource"])
coh = coh[coh["hospitalid"].isin(DEV)]
TR, _ = oc.load_transfer(RES02 / "preds", "x_only", FOLDS)
cal = oc.hospital_calibration(TR)
D_SITE = pd.read_csv(RES06B / "descripteurs_site.csv", index_col=0); D_SITE.index = D_SITE.index.astype(int)
D_M = D_SITE.join(cal[["mean_lp", "sd_lp"]])
ARR = osz.DevArrays(DATA_DIR, st, coh, TR, D_SITE, FROZEN)
HOSP = cal.index.to_numpy()
assert set(ARR.H) == set(HOSP) == set(DEV) and not (set(ARR.H) & LOCKED)
ZF = np.array([ARR.site(h)[0] for h in HOSP])
CONTROLES = []
oc.check("Descripteurs recalculés identiques au 06b", np.allclose(ZF, D_M.loc[HOSP, FROZEN].to_numpy(float), atol=1e-9, equal_nan=True), CONTROLES)
man = pd.read_csv(DATA_DIR / "clients" / "manifest.csv", usecols=["hospitalid", "n_train", "n_val", "n_test"])
ml = man[man["hospitalid"].isin(LOCKED)]
PAIRS = np.column_stack([ml["n_train"], ml["n_val"] + ml["n_test"]]).astype(float)
assert all(r["ok"] for r in CONTROLES)
print(f"{len(HOSP)} hôpitaux ; {len(FROZEN)} descripteurs ; effectifs verrouillés médians train {np.median(PAIRS[:, 0]):.0f}, évaluation {np.median(PAIRS[:, 1]):.0f}")

OK     Descripteurs recalculés identiques au 06b
60 hôpitaux ; 44 descripteurs ; effectifs verrouillés médians train 163, évaluation 70


## Étape 3 — Schéma S1 : plis par hôpital (non-dommage)

In [4]:
# ============================================================
# ÉTAPE 3 — S1
# ============================================================
t0 = time.time()
FS1 = [cal["fold"].to_numpy()] + osz.make_partitions(len(HOSP), len(FOLDS), N_PART, SEED)
PER1, INF1 = ov.run_scheme(ARR, cal, FROZEN, FS1, seed=SEED)
print(f"S1 : {time.time() - t0:.0f} s")
TAB1 = ov.scheme_table(PER1[0]); B1 = ov.boot_max(PER1[0], N_BOOT, SEED)
display(TAB1.round(5)); print(B1)

  répétition 1/11 terminée
  répétition 2/11 terminée
  répétition 3/11 terminée
  répétition 4/11 terminée
  répétition 5/11 terminée
  répétition 6/11 terminée
  répétition 7/11 terminée
  répétition 8/11 terminée
  répétition 9/11 terminée
  répétition 10/11 terminée
  répétition 11/11 terminée
S1 : 259 s


,Δℓ (par patient),Δℓ (moyenne par hôpital),MSE débiaisé
E0,0.00000,0.00000,0.09856
E1,-0.00004,-0.00005,0.10289
v1,0.00122,0.00160,0.05925
v1_retreci,0.00102,0.00130,0.06666
v1_borne,0.00127,0.00166,0.05679
v1_debruite,0.00116,0.00152,0.06090
GP_sans_debruitage,0.00100,0.00132,0.06636
v2,0.00099,0.00130,0.06707


{'D_max': np.float64(-0.0002795709835660312), 'D_max_bas': np.float64(-0.0005596644961696827), 'D_max_haut': np.float64(-3.6152295906563354e-05), 'meilleur_comparateur': 'v1_borne', 'D_v1': np.float64(-0.00023049539333669052), 'D_v1_bas': np.float64(-0.0005073007595763576), 'D_v1_haut': np.float64(2.2386434108598212e-05)}


## Étape 4 — Schémas S2 et S3 : retrait d'un groupe de case-mix, taille complète et taille des petits hôpitaux

In [ ]:
# ============================================================
# ÉTAPE 4 — S2 ET S3
# ============================================================
CASE = [c for c in FROZEN if c.startswith(ov.CASE_PREFIX)]
Xc = D_M.loc[HOSP, CASE].to_numpy(float)
Xc = np.where(np.isfinite(Xc), Xc, np.nanmean(Xc, 0))
Xc = (Xc - Xc.mean(0)) / np.where(Xc.std(0) > 1e-9, Xc.std(0), 1.0)
FS2 = [ov.kmeans(Xc, K_CLUST, SEED + s) for s in range(N_SEEDS)]
print("tailles des groupes (graine 1) :", np.bincount(FS2[0]).tolist())
t0 = time.time()
PER2, INF2 = ov.run_scheme(ARR, cal, FROZEN, FS2, pairs=PAIRS, modes=(0, N_DRAW), seed=SEED)
print(f"S2 et S3 : {time.time() - t0:.0f} s")
TAB2, TAB3 = ov.scheme_table(PER2[0]), ov.scheme_table(PER2[N_DRAW])
B2, B3 = ov.boot_max(PER2[0], N_BOOT, SEED), ov.boot_max(PER2[N_DRAW], N_BOOT, SEED)
display(TAB2.round(5)); print(B2); display(TAB3.round(5)); print(B3)

tailles des groupes (graine 1) : [24, 5, 22, 4, 5]
  répétition 1/10 terminée
  répétition 2/10 terminée
  répétition 3/10 terminée
  répétition 4/10 terminée
  répétition 5/10 terminée
  répétition 6/10 terminée
  répétition 7/10 terminée
  répétition 8/10 terminée


## Étape 5 — Décision (règle fixée par la spécification), diagnostics et rapport

In [ ]:
# ============================================================
# ÉTAPE 5 — DÉCISION, DIAGNOSTICS, RAPPORT
# ============================================================
R_S3 = bool(B3["D_max_bas"] > 0)
R_S2 = bool(B2["D_max"] > 0)
R_S1 = bool(B1["D_v1_bas"] > -0.0005)
RETENUE = R_S3 and R_S2 and R_S1
DEC = pd.DataFrame([
    {"condition": "S3 : borne basse IC de D > 0", "valeur": f"D = {B3['D_max']:.5f} [{B3['D_max_bas']:.5f} ; {B3['D_max_haut']:.5f}] (meilleur comparateur : {B3['meilleur_comparateur']})", "vraie": R_S3},
    {"condition": "S2 : D > 0", "valeur": f"D = {B2['D_max']:.5f} [{B2['D_max_bas']:.5f} ; {B2['D_max_haut']:.5f}] (meilleur comparateur : {B2['meilleur_comparateur']})", "vraie": R_S2},
    {"condition": "S1 : borne basse IC de Δℓ(v2) − Δℓ(v1) > −0,0005", "valeur": f"{B1['D_v1']:.5f} [{B1['D_v1_bas']:.5f} ; {B1['D_v1_haut']:.5f}]", "vraie": R_S1},
]).set_index("condition")
display(DEC)
print("DÉCISION :", "OP-Cal v2 RETENUE" if RETENUE else "OP-Cal v2 NON RETENUE")

INF = pd.concat([INF1.assign(schema="S1"), INF2.assign(schema="S2/S3")])
DIAG = INF.groupby("schema").agg(plis=("pli", "size"), gp_non_converge=("gp_converge", lambda s: int((~s.astype(bool)).sum())),
                                 gp_aux_bornes=("gp_borne", "sum"), kappa_median=("kappa", "median"),
                                 tau2_med=("tau2", "median"), s2L_med=("s2L", "median"), s2R_med=("s2R", "median"), ellR_med=("ellR", "median"))
display(DIAG.round(4))
oc.check("Aucun hôpital verrouillé analysé", not (set(ARR.H) & LOCKED), CONTROLES)
oc.check("Module opcal_gp inchangé pendant l'exécution", sha256(SRC_DIR / "opcal_gp.py") == GP_SHA, CONTROLES)
CONTROLES_DF = pd.DataFrame(CONTROLES)

def md_table(df, digits=5):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    esc = lambda x: str(x).replace("|", "\\|")
    head = "| " + " | ".join([esc(df.index.name or "")] + [esc(c) for c in df.columns]) + " |"
    return "\n".join([head, "|" + "---|" * (len(df.columns) + 1)] + ["| " + " | ".join([esc(i)] + [esc(v) for v in r]) + " |" for i, r in zip(df.index, df.to_numpy())])

rapport = f"""# Rapport 10 — validation interne d'OP-Cal v2 (mortalité, 60 hôpitaux de développement)

Généré le {datetime.now():%Y-%m-%d %H:%M}. Spécification 0.3 ; `opcal_gp.py` `{GP_SHA[:16]}…` ; plan `plan_10.json` du {PLAN['date']}.
Aucune donnée des 96 hôpitaux verrouillés, hormis leurs effectifs du manifest (taille des sous-échantillons).

## Décision : **{"OP-Cal v2 RETENUE" if RETENUE else "OP-Cal v2 NON RETENUE"}**

{md_table(DEC)}

## S1 — plis par hôpital (taille complète)

{md_table(TAB1)}

## S2 — retrait d'un groupe de case-mix (taille complète)

{md_table(TAB2)}

## S3 — retrait d'un groupe de case-mix, taille des petits hôpitaux

{md_table(TAB3)}

Δℓ : gain de log-vraisemblance par patient contre E0 (une observation par hôpital). Le MSE débiaisé n'a de sens qu'à taille complète (sites ≥ 10 événements).

## Diagnostics d'estimation

{md_table(DIAG.reset_index().set_index('schema'), 4)}

## Contrôles

{md_table(CONTROLES_DF.set_index('controle'))}
"""
(RES_DIR / "rapport_10.md").write_text(rapport, encoding="utf-8")
for k, t in {"S1": TAB1, "S2": TAB2, "S3": TAB3}.items():
    t.to_csv(RES_DIR / f"methodes_{k}.csv")
np.savez(RES_DIR / "par_hopital.npz", **{f"{k}_{x}": PER[0 if k != 'S3' else N_DRAW][x] for k, PER in (("S1", PER1), ("S2", PER2), ("S3", PER2)) for x in ("G", "n", "SQ")})
INF.to_csv(RES_DIR / "diagnostics_plis.csv", index=False)
print(f"Écrit : {RES_DIR / 'rapport_10.md'}")